# 实践教程

**Zvec** 是一款面向开发者的*轻量级*、*高性能*、*进程内*、*开源*向量数据库，适合构建快速、可靠的相似度搜索能力。

你可以把它作为*独立向量数据库*使用，也可以把它嵌入到现有数据库系统中，作为专用的*向量搜索引擎*。

Zvec 开箱即用地支持索引、过滤和分组查询，并支持稠密/稀疏向量、多种索引策略（如 `HNSW` 和 `IVF`）、量化以及动态字段 Schema，所有能力都可以通过简洁的 Python API 使用。

本 Notebook 会通过可运行示例，帮助你快速、有效地集成 Zvec。

- 如果你还不熟悉向量搜索中的概念，例如 embedding、向量索引、量化等，请先阅读文档中的[核心概念](https://zvec.org/zh/docs/db/concepts/)章节。
- 如需详细 API 参考、配置项和使用模式，请访问 [Zvec 文档](https://zvec.org/zh/docs/)。

## 目录

1. [安装](#安装)
1. [全局配置](#全局配置)
1. [创建 Collection](#创建-collection)
1. [插入数据](#插入数据)
1. [使用向量搜索查询](#使用向量搜索查询)
1. [进阶功能](#进阶功能)
1. [销毁 Collection](#销毁-collection)

## 安装

Zvec 已作为标准 Python 包发布在 [PyPI](https://pypi.org/project/zvec/) 上。对大多数用户来说，这是最快、最简单的上手方式。

```python
# - 在 Jupyter Notebook 中：使用 %pip 语法安装到当前 Python 环境
# - 在终端或普通 Python 脚本中：直接运行 pip install（不带 %）
pip install zvec
```

In [ ]:
%pip install zvec

In [ ]:
# 安装完成后，验证是否可以正常使用：
import zvec

print("Zvec 版本：", zvec.__version__)

## 全局配置

在使用 Zvec 前，我们先进行全局配置：将日志类型设置为 `console`，日志级别设置为 `WARN`。
这样 Notebook 输出中只会展示重要 warning 及更严重的信息，既保持整洁，也能及时提醒潜在问题。

In [ ]:
# 只需要运行一次。
zvec.init(log_type=zvec.LogType.CONSOLE, log_level=zvec.LogLevel.WARN)

## 创建 Collection

在 Zvec 中，数据被组织为 **Collection**，也就是用于管理相关数据、索引和存储设置的逻辑容器。

你可以把 Collection 理解为关系型数据库中的表，不过它针对向量搜索和混合查询做了优化。

每个 Collection 都必须通过明确的 **Schema** 定义，Schema 包含：

1. `name`：Collection 的标识符。
1. `fields`：标量字段列表。
1. `vectors`：向量字段列表。

创建 Collection 时，可以使用 `create_and_open()` 函数并传入 `CollectionSchema`。

如果你想了解更多细节，可以查看[完整文档](https://zvec.org/zh/docs/db/collections/create/)。不过现在也可以先跳过，直接跟着这个 Notebook 运行。


### 示例：创建用于图片搜索的 Collection

我们将创建一个名为 `image_search` 的 Collection，其中包含：
- 一个**标量**字段 `base64_image`：用于展示图片
- 一个**向量**字段 `embedding`：用于相似度搜索

下面是定义并创建这个 Collection 的方式：

In [ ]:
# 使用 FieldSchema 定义标量字段
image_encoding = zvec.FieldSchema(
    name="base64_image",
    data_type=zvec.DataType.STRING,  # 字符串类型
)

# 使用 VectorSchema 定义向量字段
image_embedding = zvec.VectorSchema(
    name="embedding",
    data_type=zvec.DataType.VECTOR_FP32,  # float32 精度的稠密向量
    dimension=1024,  # embedding 向量维度
    # 使用 HNSW 向量索引，并采用余弦相似度
    index_param=zvec.HnswIndexParam(metric_type=zvec.MetricType.COSINE),
)

# 定义 Collection Schema
collection_schema = zvec.CollectionSchema(
    name="image_search",
    fields=[image_encoding],  # 标量字段列表（这里暂时只定义一个）
    vectors=[image_embedding],  # 向量字段列表（这里暂时只定义一个）
)

# 创建 Collection
collection = zvec.create_and_open(
    path="./image_search",  # Collection 存储路径（你可以自行修改）
    schema=collection_schema,
)

现在，你应该能在当前工作目录下看到一个名为 **image_search** 的新目录（如果你修改了配置，则目录名可能不同）。

> ⚠️ **注意**：
> - `create_and_open()` 会创建一个**新的** Collection；如果指定路径已经存在，它会失败。
        如果看到类似 “path already exists” 的错误，请删除该 Collection 目录，并重启 Notebook kernel 后重新开始。
> - 在 Jupyter Notebook 中很容易重复运行单元格，但 `create_and_open()` 只适合第一次运行。再次运行时，Collection 已经存在，因此会失败。
> - 如果要加载已有 Collection，请使用 `open()` 函数。
>       本教程只演示从空 Collection 开始的流程，不展开 `open()` 函数；详情请参考[文档](https://zvec.org/zh/docs/db/collections/open/)。

我们使用 [HNSW 索引](https://zvec.org/zh/docs/db/concepts/vector-index/hnsw-index/)来高效搜索 `embedding` 向量。

**请始终确保向量索引设置（例如相似度度量和维度）与你的 embedding 模型保持一致**。

在这个示例中，我们使用余弦相似度和 1024 维向量索引，因为底层 embedding 模型以余弦相似度训练，并输出 1024 维向量。
在你的实际场景中，请按照所选 embedding 模型的规格进行设置。

查看 Collection Schema：

In [ ]:
print(collection.schema)

如需了解 Schema 格式的详细说明，请参考[文档](https://zvec.org/zh/docs/db/collections/inspect/#collection-schema)。

查看 Collection 统计信息：

In [ ]:
print(collection.stats)
# 因为 Collection 目前为空，所以 doc_count 应该为 0

## 插入数据

Collection 创建完成后，就可以开始插入数据。

在典型使用场景中，你会先拥有**原始数据**，例如图片、文本或其他非结构化内容。原始数据会先经过 [embedding 模型](https://zvec.org/zh/docs/db/concepts/vector-embedding/#什么是-embedding-模型)处理，转换为[向量 embedding](https://zvec.org/zh/docs/db/concepts/vector-embedding/)。除了 embedding，你通常还会存储一些**标量字段**（例如类别等），用于过滤、召回或混合查询。

为了让这个 Notebook **自包含**且**易于运行**，我们已经预先生成了一个小型 **JSONL** 示例数据集。原始图片来自 [ImageNet-Val5k](https://modelscope.cn/datasets/iic/imagenet-val5k-image) 数据集，图片 embedding 使用 **Qwen2.5-VL-Embedding** 多模态模型提前计算完成。

**JSONL** 文件中的每个 document 都包含：

1. `id`：document 的唯一标识。
1. `image_encoding`：原始图片的 base64 编码字符串。
1. `image_embedding`：图片的预计算向量 embedding。
1. `category`：可选的类别标签，例如 `animal`、`vehicle`。

接下来，我们会加载这份可直接使用的数据，并通过 `insert()` 方法插入到 Collection 中。

> 注意：继续之前，你可能需要安装几个用于处理数据的依赖。
> 如果你在一个干净环境中运行此 Notebook，请运行下面的单元格，确保环境准备完毕。

In [ ]:
%pip install Pillow
%pip install matplotlib

### 加载并查看数据

现在先加载示例数据，并快速查看它的结构。

In [ ]:
import walkthrough_utils

# 使用提供的工具函数加载预生成示例数据
data = walkthrough_utils.load_jsonl("./data.jsonl")

`data` 是一个字典列表（`list[dict]`），每个字典表示一个 document，包含以下键值对：

1. `id`
1. `image_encoding`
1. `image_embedding`
1. `category`（可选）

查看部分样例：

In [ ]:
# 查看数据集的结构和大小
print("第一个 document 的结构：", data[0].keys())
print(f"加载的 document 总数：{len(data)}")

# 预览一个示例 document（可以修改 index 来探索其他样例）
example_index = 2

# 获取原始 document
raw_doc = data[example_index]
print(f"\n展示数据集中的 image[{example_index}]：")

# 使用提供的工具函数，从 base64 字符串展示图片
walkthrough_utils.display_image_from_base64(raw_doc["image_encoding"])
print(f"ID：{raw_doc['id']}")
print(f"图片 embedding 维度：{len(raw_doc['image_embedding'])}")
print("图片 embedding：", raw_doc["image_embedding"])
if "category" in raw_doc:
    print("类别：", raw_doc["category"])

> ⚠️ **注意**：
> - `image_encoding` 字段包含图片的 base64 编码字符串，非常长且不适合直接阅读，因此我们不会直接打印它。
        这里会使用辅助函数 `display_image_from_base64()` 渲染真实图片。
> - `image_embedding` 是一个 1024 维向量。直接打印会得到很长的**浮点数列表**，单独查看并不直观；这里仅展示片段用于完整性说明。
        实际开发中，你通常会以编程方式使用这些 embedding（例如做相似度搜索），而不是肉眼阅读。

### 将数据插入 Collection

加载并查看示例数据后，下一步是将它插入到 Collection 中。

这个过程会把每条原始记录转换为 `Doc` 对象，然后调用 `insert()` 方法将这些 document 加入 Collection。

`Doc` 表示 Collection 中的单个 [document](https://zvec.org/zh/docs/db/concepts/data-modeling/#documents)，由三个主要部分组成：

1. `id`：document 的唯一标识。
1. `fields`：命名标量（非向量）字段字典。
1. `vectors`：命名向量 embedding 字典。

`fields` 和 `vectors` 都使用字典结构：每个 key 对应 Collection Schema 中定义的字段名或向量名，每个 value 保存实际数据。

这种设计允许你在单个 document 中包含多个字段或多个向量。你提供的键值对必须**符合创建 Collection 时指定的 Schema**。

在这个简单示例中，Collection Schema 只定义了一个标量字段（`base64_image`，字符串类型）和一个向量（`embedding`，浮点数列表），所以每个 document 只包含这两个条目。

In [ ]:
# 遍历数据集中的每条原始 document
for raw_doc in data:
    # 使用原始 document 中的 id、base64 编码图片和图片 embedding 创建新的 Doc 实例
    id: str = raw_doc["id"]  # 字符串
    image: str = raw_doc["image_encoding"]  # 字符串
    vector: list[float] = raw_doc["image_embedding"]  # 浮点数列表
    doc = zvec.Doc(
        id=id,
        fields={
            "base64_image": image,
        },
        vectors={
            "embedding": vector,
        },
    )
    # 将新创建的 document 插入 Collection
    result = collection.insert(doc)
    # 检查插入是否成功
    if not result.ok():
        print(result)
        break

> ⚠️ **注意**：
> 上面的代码执行的是[**插入**](https://zvec.org/zh/docs/db/data-operations/insert/)，不是 [upsert](https://zvec.org/zh/docs/db/data-operations/upsert/)。这意味着如果 Collection 中已经存在相同 id 的 document，该操作会失败。
>
> 如果你看到类似 `doc_id[*] already exists in collection` 的错误，不用紧张！这通常表示该 document（或具有相同 `id` 的 document）已经被插入过了。
        在开发或重复运行导入脚本时，这很常见。

### 再次查看 Collection 统计信息

我们再检查一次 Collection 统计信息。这时你应该会看到 Collection 中包含 **130** 个 document，与数据集中的记录总数一致。

你可能会注意到 `index_completeness` 为 0。这表示这些 document 的 HNSW 索引**尚未构建完成**。
在索引完成前，向量搜索会回退到暴力搜索；这会慢一些，但仍然可用。暂时不用担心，后面会进一步说明。

In [ ]:
print(collection.stats)

## 使用向量搜索查询

现在数据已经成功插入 Collection，我们可以执行**向量相似度搜索**，召回语义相关的结果。

Collection 中的图片 embedding 由 **Qwen2.5-VL-Embedding** 模型生成。
由于该模型是**多模态**模型，它会把图片和文本映射到同一个语义空间中，也就是说，你既可以用*图片*作为查询，也可以用*文本描述*作为查询。

为了让这个 Notebook **自包含**且**易于运行**，我们已经为几个示例查询提前计算了 embedding，并保存为 **JSONL** 文件。

接下来会加载这些预计算查询向量并直接用于搜索。如果你感兴趣，也可以尝试使用 **Qwen2.5-VL-Embedding** 模型生成自己的查询 embedding。

### 文本查询

我们先从**文本查询**开始。

例如，要查找 “cute dog” 的图片，通常会：

1. 使用 **Qwen2.5-VL-Embedding** 模型将该短语编码为向量，然后
1. 在 Collection 中执行向量搜索。

在这个 Notebook 中，我们会跳过编码步骤，直接使用预计算 embedding。

下面是加载预生成文本查询的方式。每一项都是一个字典，包含：

- `text`：原始查询字符串
- `embedding`：对应的向量表示

In [ ]:
# 使用提供的工具函数加载预生成文本查询
text_queries = walkthrough_utils.load_jsonl("./text_queries.jsonl")

# 查看文本查询
print(f"加载的文本查询总数：{len(text_queries)}")
print("第一个文本查询的结构：", text_queries[0].keys())

# 查看第一个文本查询
query_index = 0  # 可以修改 index 来查看不同查询
print(f"\n索引 {query_index} 对应的文本查询是：")
print(f"文本：{text_queries[query_index]['text']}")
print("Embedding：", text_queries[query_index]["embedding"])

第一个文本查询是 **“Cute dog”**，我们用它在 Collection 中执行一次**向量相似度搜索**：

In [ ]:
print(f"第一个文本查询：{text_queries[0]['text']}")

result = collection.query(
    zvec.VectorQuery(
        field_name="embedding",  # Collection 中用于比较的向量字段
        vector=text_queries[0]["embedding"],  # “cute dogs” 查询的 embedding
    ),
    topk=3,  # 召回最相似的 3 张图片
    include_vector=False,  # 不返回完整向量，以减少返回数据量
)

print("\n查询结果：\n")
print(result)

现在，我们已经为查询 **“Cute dog”** 召回了相关性最高的 3 个 document。

返回的 `result` 是 `Doc` 对象列表，其中每个 `Doc` 包含：

- `id`：document 的唯一标识（也就是插入时提供的 id）。
- `fields`：字段名和值的字典。在本例中，它包含 `base64_image` 字符串。
- `vectors`：向量名和值的字典。这里为空，因为查询时没有指定 `include_vector=True`。

由于原始 base64 字符串不适合阅读，我们将其解码并展示真实图片。
通过遍历 `result` 列表，可以方便地展示每张图片及其对应的 `id`。

In [ ]:
# 使用提供的工具函数，从 base64 字符串展示每张图片
for idx, doc in enumerate(result):
    print(f"\n展示第 {idx + 1} 张图片，ID：{doc.id}")
    walkthrough_utils.display_image_from_base64(doc.fields["base64_image"])

看起来我们找到了几张很可爱的匹配图片！🐾 🐶

接下来定义一个辅助函数，用来运行查询并展示相似度最高的 3 张图片，方便探索更多文本查询。

In [ ]:
def display_similar_images_by_text_query(query_index: int):
    text_query = text_queries[query_index]
    # 打印当前使用的文本查询
    print(f"文本查询[{query_index}]：{text_query['text']}")
    # 执行向量相似度搜索
    result = collection.query(
        zvec.VectorQuery(
            field_name="embedding",  # Collection 中用于比较的向量字段
            vector=text_query["embedding"],  # 文本 embedding
        ),
        topk=3,  # 召回最相似的 3 张图片
        include_vector=False,  # 不返回完整向量，以减少返回数据量
    )
    # 展示召回图片
    for idx, doc in enumerate(result):
        print(f"\n展示第 {idx + 1} 张图片，ID：{doc.id}")
        walkthrough_utils.display_image_from_base64(doc.fields["base64_image"])

现在试试第二个查询：**“Find images that have both a dog and a car.”**

In [ ]:
# 可以修改 query_index 来探索其他查询
display_similar_images_by_text_query(query_index=1)

💡 **说明**：

你可能会发现第二张和第三张图片并不完全符合描述。
这是因为我们的示例数据集**太小**，其中**只有一张图片同时包含狗和车**。
其他图片虽然不是完全匹配，但仍然与查询语义存在一定关联。

欢迎继续尝试其他查询，祝你探索愉快！🌄🚗🐕

请记住，**top-1** 图片通常应该与文本查询最接近，而后面的图片可能会因为数据库规模较小而略微偏题。不过你仍然能看到它们与查询之间存在一些主题或上下文关联。

> ✅ 在真实生产场景中，数据库通常包含数百万甚至数十亿条数据，向量相似度搜索会更加精确、相关。

### 图片查询

到目前为止，我们一直在用文本搜索图片。现在试试**用图片作为查询**！

我们已经为几个示例查询图片提前计算了 embedding，并保存为 **JSONL** 文件。每一项包含：

- `encoding`：原始图片的 base64 编码字符串
- `embedding`：对应的向量表示

下面加载这些图片查询，并尝试运行一次。

In [ ]:
# 使用提供的工具函数加载预生成图片查询
image_queries: list[dict] = walkthrough_utils.load_jsonl("./image_queries.jsonl")

print(f"已加载 {len(image_queries)} 个图片查询。")


# 定义辅助函数：展示查询图片，并从 Collection 中召回相似图片
def display_similar_images_by_image_query(query_index: int):
    image_query = image_queries[query_index]
    # 展示当前查询图片
    print(f"图片查询[{query_index}]：")
    walkthrough_utils.display_image_from_base64(image_query["encoding"])
    # 执行向量相似度搜索
    result = collection.query(
        zvec.VectorQuery(
            field_name="embedding",  # Collection 中用于比较的向量字段
            vector=image_query["embedding"],  # 图片 embedding
        ),
        topk=3,  # 召回最相似的 3 张图片
        include_vector=False,  # 不返回完整向量，以减少返回数据量
    )
    # 展示召回图片
    print(f"\n召回了 {len(result)} 张相似图片：")
    for idx, doc in enumerate(result):
        print(f"\n展示第 {idx + 1} 张图片，ID：{doc.id}")
        walkthrough_utils.display_image_from_base64(doc.fields["base64_image"])

> 💡 **说明**：你可以修改 `query_index` 来探索不同的图片查询。
> **top-1** 图片通常应该与查询图片最接近，后面的图片可能会因为数据库规模较小而略微偏题。

In [ ]:
# 可以修改 query_index
display_similar_images_by_image_query(0)

## 进阶功能

本教程已经覆盖了创建 Collection 和执行向量查询的基础流程。不过，Zvec 还提供了许多进阶能力，用于支持更复杂的场景：

- **带过滤条件的向量查询**
- **稠密向量和稀疏向量混合支持**
- **单个 document 支持多个标量字段和多个向量 embedding**
- **多种索引类型和灵活的索引策略**

如需深入了解这些功能，包括配置选项和性能建议，请参考[官方文档](https://zvec.org/zh/docs/)。

## 销毁 Collection

当你完成实验，或不再需要某个 Collection 时，可以将其永久删除以释放存储空间。

Zvec 提供了一个简单但**不可逆**的方法：

```python
collection.destroy()
```

该操作会**删除与 Collection 关联的所有数据**。删除后无法恢复。

> ⚠️ 警告：`destroy()` 是永久操作。调用前请再次确认你不再需要这些数据。

In [ ]:
# 永久删除
collection.destroy()

print("✅ Collection 已成功删除。")